# First-wall damage and gas production

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nukehub-dev/nucleide/blob/main/notebooks/first-wall-damage.ipynb)

Neutron bombardment displaces atoms (dpa) and breeds helium/hydrogen gas (appm) in first-wall materials. This notebook folds a synthetic three-group flux with caller-supplied response cross sections through `nucleide.damage`, then propagates a joint uncertainty block through the He/dpa ratio — including the 68% interval that holds where a symmetric spread cannot.

> On Colab, run `%pip install "nucleide>=0.16.0"` first. Everything below uses synthetic round numbers — no evaluated nuclear data is bundled or fetched.

In [ ]:
from nucleide.damage import arc_dpa, fold_uq, gas_appm, he_dpa_ratio, he_dpa_ratio_uq, nrt_dpa

# Three-group grid (MeV boundaries); flux is per-group integrated flux [n/cm2/s],
# responses are group cross sections [barns] from your own pipeline.
bounds = [0.0, 0.1, 1.0, 20.0]
flux = [1.0e12, 2.0e12, 4.0e12]
dpa_xs = [100.0, 200.0, 50.0]
he_xs = [0.5, 1.0, 0.25]
seconds = 3.0 * 365.25 * 24 * 3600  # three full-power years

dpa = nrt_dpa(flux, dpa_xs, bounds, seconds)
arc = arc_dpa(flux, dpa_xs, bounds, seconds)
appm = gas_appm(flux, he_xs, bounds, seconds)
ratio = he_dpa_ratio(flux, he_xs, dpa_xs, bounds, seconds)
print(f"NRT-dpa   : {dpa:.6g}")
print(f"arc-dpa   : {arc:.6g}")
print(f"He        : {appm:.6g} appm")
print(f"He/dpa    : {ratio:.6g} appm/dpa")
assert ratio == appm / dpa  # the ratio is defined by the two folds

## Uncertainty through the folds

Perturb the stacked `[flux, response]` vector with a seeded multivariate-normal block (relative perturbations). Linear folds carry exact expectations; the He/dpa ratio is formed per draw, with the draw mean gated against the bias-corrected expectation and the draw 68% interval gated against the Fieller-construction quantiles.

In [ ]:
# Uncorrelated 2% block on flux + dpa response (dimension 2G = 6).
mean = [0.0] * 6
cov = [[0.0004 if i == j else 0.0 for j in range(6)] for i in range(6)]
out = fold_uq("nrt_dpa", flux, dpa_xs, bounds, seconds, mean, cov, 4000, 20260915, 5.0)
print(f"mean {out['mean']:.6g} vs expected {out['expected']:.6g}  passed={out['passed']}")
assert out["passed"]

# Joint [flux | He | dpa] block, +-10% uncorrelated on both responses.
mean3 = [0.0] * 9
cov3 = [[0.0] * 9 for _ in range(9)]
cov3[4][4] = 0.01
cov3[8][8] = 0.01
ruq = he_dpa_ratio_uq(flux, he_xs, dpa_xs, bounds, seconds, mean3, cov3, 20000, 20260915, 5.0)
print(f"nominal {ruq['nominal']:.6g}  mean {ruq['mean']:.6g} +- {ruq['std']:.6g}")
print(f"median {ruq['q50']:.6g}  68% interval [{ruq['q16']:.6g}, {ruq['q84']:.6g}]")
print(
    f"moment gate passed={ruq['passed']} (honestly fails here)"
    f"  interval gate passed={ruq['quantiles_passed']}"
)
assert ruq["quantiles_passed"]
assert ruq["q16"] < ruq["q50"] < ruq["q84"]  # ordered, skewing upward